# 03 Statistics

In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("C:/Users/Siddhesh/Downloads/bank-customer-churn/data/processed/Bank_churn_prediction.csv")

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,AgeBand,HasZeroBalance,BalaceBand,TenureBucket,ProductGroup,CrediBand
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,40-49,1,Zero,2-4,1,580-669
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,40-49,0,1-99k,2-4,1,580-669
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,40-49,0,100k+,8-10,3+,<580
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,30-39,1,Zero,2-4,2,670-739
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,40-49,0,100k+,2-4,1,740+


In [16]:
pip install scipy 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from scipy import stats

In [18]:
def cramers(col):
    ct = pd.crosstab(df[col], df["Exited"])

    chi2, p, dof, _ = stats.chi2_contingency(ct)

    v = np.sqrt(
        chi2 / (ct.values.sum() * (min(ct.shape) - 1))
    )

    return pd.Series({
        "p_value": p,
        "cramers_v": v
    })

In [19]:
print(df.columns.tolist())

['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'AgeBand', 'HasZeroBalance', 'BalaceBand', 'TenureBucket', 'ProductGroup', 'CrediBand']


In [23]:
cats = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "IsActiveMember",
    "HasCrCard",
    "AgeBand"
]

results = pd.DataFrame({
    c: cramers(c) for c in cats
}).T.sort_values("cramers_v", ascending=False)

print(results)

                      p_value  cramers_v
NumOfProducts    0.000000e+00   0.387767
AgeBand         1.899149e-276   0.358144
Geography        3.830318e-66   0.173567
IsActiveMember   8.785858e-55   0.155880
Gender           2.248210e-26   0.106263
HasCrCard        4.923724e-01   0.006865


In [12]:
for c in [
    "Age",
    "Balance",
    "CreditScore",
    "EstimatedSalary",
    "Tenure"
]:
    a = df.loc[df.Exited == 1, c]
    b = df.loc[df.Exited == 0, c]

    print(
        c,
        "p =", stats.mannwhitneyu(a, b).pvalue,
        "| median churned =", a.median(),
        "| median retained =", b.median()
    )

Age p = 3.2104014319371675e-230 | median churned = 45.0 | median retained = 36.0
Balance p = 1.1158399379149222e-28 | median churned = 109349.29 | median retained = 92072.68
CreditScore p = 0.01986866117272834 | median churned = 646.0 | median retained = 653.0
EstimatedSalary p = 0.22705150930153783 | median churned = 102460.84 | median retained = 99645.04
Tenure p = 0.16219376122918827 | median churned = 5.0 | median retained = 5.0


# Statistical Testing — Customer Churn

## 1. Categorical Analysis (Cramér's V)

| Feature | Cramér's V | Finding |
|---|---:|---|
| NumOfProducts | 0.388 | Strongest association |
| AgeBand | 0.358 | Strong association |
| Geography | 0.174 | Moderate/weak |
| IsActiveMember | 0.156 | Weak |
| Gender | 0.106 | Weak |
| HasCrCard | 0.007 | Negligible |

## 2. Numerical Analysis (Mann–Whitney U)

| Feature | Median Churned | Median Retained | Result |
|---|---:|---:|---|
| Age | 45 | 36 | Significant |
| Balance | 109,349.29 | 92,072.68 | Significant |
| CreditScore | 646 | 653 | Significant |
| EstimatedSalary | 102,460.84 | 99,645.04 | Not significant |
| Tenure | 5 | 5 | Not significant |

## 3. Key Findings

- `NumOfProducts` and `AgeBand` have the strongest categorical associations with churn.
- Churned customers are older and have higher median balances.
- CreditScore has a statistically significant but small difference.
- EstimatedSalary and Tenure show no significant difference.
- `HasCrCard` has negligible association with churn.

## 4. Conclusion

Age, number of products, and balance are promising features for further churn analysis. These findings show associations, not causation. Features will be evaluated further using cross-validated PR-AUC.